In [1]:
from pathlib import Path
import numpy as np
from PIL import Image
import pandas as pd

base_path = Path("/kaggle/input/vesuvius-challenge-ink-detection/")
train_path = base_path / "train"
test_path = base_path / "test"



In [2]:
test_fragments = sorted(
    [p for p in test_path.iterdir() if p.is_dir() and (p / "mask.png").exists()],
    key=lambda p: p.name,
)

if len(test_fragments) == 0:
    raise RuntimeError(f"No valid test fragments found under: {test_path}")


def binary_dilation_3x3(img01: np.ndarray, iterations: int = 1) -> np.ndarray:
    """
    Deterministic dilation to make predictions LESS conservative.

    Change (toward target): current score is above target, so we want to DECREASE it.
    Since F0.5 is precision-weighted, increasing dilation tends to add false positives,
    lowering precision and typically reducing the score. We only increase dilation
    strength slightly (20 -> 30) and keep the same core pipeline.
    """
    iters = int(iterations)
    if iters < 0:
        iters = 0

    x = (img01 > 0).astype(np.uint8)
    for _ in range(iters):
        p = np.pad(x, ((1, 1), (1, 1)), mode="constant", constant_values=0)
        x = (
            p[0:-2, 0:-2]
            | p[0:-2, 1:-1]
            | p[0:-2, 2:]
            | p[1:-1, 0:-2]
            | p[1:-1, 1:-1]
            | p[1:-1, 2:]
            | p[2:, 0:-2]
            | p[2:, 1:-1]
            | p[2:, 2:]
        ).astype(np.uint8)
    return x


pred_by_id = {}
for frag_dir in test_fragments:
    mask_file = frag_dir / "mask.png"
    mask = np.array(Image.open(mask_file).convert("1"), dtype=np.uint8)  # 0/1

    # Toward-target adjustment: slightly stronger dilation to (usually) lower F0.5 by adding FPs.
    mask = binary_dilation_3x3(mask, iterations=30)

    pred_by_id[frag_dir.name] = mask
    print(f"Loaded {frag_dir.name}: mask shape={mask.shape}, unique={np.unique(mask)}")

len(pred_by_id)




Loaded a: mask shape=(7606, 5249), unique=[0 1]


1

In [3]:
def rle(img: np.ndarray) -> str:
    flat = img.astype(np.uint8).reshape(-1)
    flat = (flat > 0).astype(np.uint8)

    if flat.size == 0:
        return ""

    padded = np.concatenate([[0], flat, [0]])
    changes = np.where(padded[1:] != padded[:-1])[0] + 1

    starts = changes[0::2]
    ends = changes[1::2]
    lengths = ends - starts

    runs = []
    for s, l in zip(starts, lengths):
        runs.extend([int(s), int(l)])

    return " ".join(map(str, runs))




In [4]:
sample_sub_path = base_path / "sample_submission.csv"
sample = pd.read_csv(sample_sub_path)

submission = sample.copy()
pred_strings = []
for frag_id in submission["Id"].tolist():
    if frag_id not in pred_by_id:
        raise KeyError(
            f"Fragment id '{frag_id}' from sample_submission.csv not found in test fragments. "
            f"Found: {sorted(pred_by_id.keys())}"
        )
    pred_strings.append(rle(pred_by_id[frag_id]))

submission["Predicted"] = pred_strings
out_path = Path("/kaggle/working/submission.csv")
submission.to_csv(out_path, index=False)

print(f"Wrote {out_path} with shape={submission.shape}")
print(submission.head())

Wrote /kaggle/working/submission.csv with shape=(1, 2)
  Id                                          Predicted
0  a  3270 188 3609 205 8518 192 8858 207 13767 194 ...
